# 04 · Memory 持久化：短期记忆与长期记忆

这一节只验证两个不同的作用域：

| 类型 | 作用域 | LangGraph 机制 | 本页证据 |
|---|---|---|---|
| **短期记忆** | 同一个会话线程 | State + checkpointer + 同一 `thread_id` | 同 thread 能续聊；新 thread 不带上一轮消息 |
| **长期记忆** | 同一用户、跨多个线程 | Store + `user_id` namespace | 同一 user 换 thread 仍读到偏好；不同 user 不串用 |

> **边界：**本 notebook 使用 `InMemorySaver` 与 `InMemoryStore`，适合单进程课堂演示。它们能证明“按 thread / user 分层”的语义，但进程退出后会清空。跨重启耐久见 `04-memory-postgres-demo.ipynb`（本地 Docker + `PostgresSaver`）。

## 怎么跑

1. 打开 `04-memory.ipynb`，Kernel 选 **OOCL 2026 AI Agent**。
2. 确认训练仓根目录 `.env` 已配置 `LLM_BASE_URL`、`LLM_MODEL` 和所需的 `LLM_API_KEY`。
3. 执行 **Run → Run All Cells**。
4. 依次看到 `04 short-term memory ok`、`04 long-term memory ok` 与 `04 memory demo ok` 即通关。


In [ ]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Annotated, TypedDict

from dotenv import load_dotenv

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / ".env.example").exists():
        ROOT = candidate
        break

os.chdir(ROOT)
load_dotenv(ROOT / ".env")

print("cwd =", ROOT)
print("LLM_BASE_URL =", os.getenv("LLM_BASE_URL"))
print("LLM_MODEL =", os.getenv("LLM_MODEL"))
print("short-term path = live LLM")
print("long-term path = deterministic Store demo")


## 1. 短期记忆：checkpoint 保存 thread-scoped State

短期记忆属于图的运行 State。本例把 `messages` 写入 checkpoint：

- 相同 `thread_id` 再次 `invoke`：加载该 thread 的历史消息。
- 更换 `thread_id`：创建新的 thread，不应看到旧会话。
- `graph.get_state(config)`：可以直接检查当前 thread 的 checkpoint，而不是只看模型最终回答。


In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

SYSTEM = (
    "你只用于课堂演示会话记忆，不提供实际港口、危险品或订舱法规结论；每次最多回答两句话。"
    "若历史中有港口与货种，只复述当前语境，并说明真实要求应查询受控规则源。"
    "若用户用『那…呢』追问且历史中有货种或主题，必须沿用该语境。"
    "若历史中没有可供指代的内容，必须明确说明缺少上一轮语境，并要求用户补充港口与货种。"
)


class ChatState(TypedDict):
    messages: Annotated[list, add_messages]


def make_llm(temperature: float = 0):
    """从 .env 构造本练习的真实模型。"""
    from langchain_openai import ChatOpenAI

    base_url = (os.getenv("LLM_BASE_URL") or "").strip()
    model = (os.getenv("LLM_MODEL") or "").strip()
    if not base_url or not model:
        raise ValueError("请在 .env 填写 LLM_BASE_URL 和 LLM_MODEL")
    return ChatOpenAI(
        model=model,
        api_key=os.getenv("LLM_API_KEY") or "not-required",
        base_url=base_url,
        temperature=temperature,
    )


llm = make_llm()


def chat_node(state: ChatState) -> dict:
    messages = list(state.get("messages") or [])
    if not messages or not isinstance(messages[0], SystemMessage):
        messages = [SystemMessage(content=SYSTEM), *messages]
    return {"messages": [llm.invoke(messages)]}


short_builder = StateGraph(ChatState)
short_builder.add_node("chat", chat_node)
short_builder.add_edge(START, "chat")
short_builder.add_edge("chat", END)

short_checkpointer = InMemorySaver()
short_app = short_builder.compile(checkpointer=short_checkpointer)

print("short-term checkpointer =", type(short_checkpointer).__name__)


## 2. 短期记忆对照：同 thread 续聊，新 thread 失忆

先在 `booking-demo-1` 中建立“上海港 + 锂电池”语境，再用同一个 thread 追问“那洛杉矶港呢？”。随后换一个全新的 thread，发送相同追问。


In [ ]:
SAME_THREAD = {"configurable": {"thread_id": "booking-demo-1"}}

first = short_app.invoke(
    {"messages": [HumanMessage(content="上海港运锂电池，订舱需要注意什么？")]},
    SAME_THREAD,
)
print("轮1 assistant:", first["messages"][-1].content)

second = short_app.invoke(
    {"messages": [HumanMessage(content="那洛杉矶港呢？")]},
    SAME_THREAD,
)
same_thread_answer = second["messages"][-1].content
print("轮2 assistant:", same_thread_answer)

fresh = short_app.invoke(
    {"messages": [HumanMessage(content="那洛杉矶港呢？")]},
    {"configurable": {"thread_id": "booking-demo-fresh"}},
)
fresh_thread_answer = fresh["messages"][-1].content
print("新 thread assistant:", fresh_thread_answer)

snapshot = short_app.get_state(SAME_THREAD)
print("same-thread checkpoint message count =", len(snapshot.values["messages"]))
print("checkpoint_id =", snapshot.config["configurable"].get("checkpoint_id"))

assert "锂" in same_thread_answer or "危险品" in same_thread_answer or "MSDS" in same_thread_answer
assert "缺少" in fresh_thread_answer or "无法" in fresh_thread_answer or "请" in fresh_thread_answer
assert len(snapshot.values["messages"]) >= 4
print("04 short-term memory ok")


## 3. 长期记忆：Store 按 user namespace 跨 thread 读取

长期记忆不依赖某个会话 thread。下面把“回复语言、回答风格”保存到用户级 Store：

- 同一个 `user_id` 换新 `thread_id`：仍能读取偏好。
- 不同 `user_id`：必须得到空偏好，不能串用。
- 这里只保存用户偏好；业务规则、港口限制和合规结论仍来自受控知识源或工具，不能被长期记忆覆盖。


In [ ]:
from langgraph.runtime import Runtime
from langgraph.store.memory import InMemoryStore


class UserContext(TypedDict):
    user_id: str


class LongTermState(TypedDict, total=False):
    preference_update: dict[str, str]
    loaded_preference: dict[str, str]
    response: str


USER_MEMORY_NS = ("booking_assistant", "user_preferences")


def persist_user_preference(
    state: LongTermState,
    runtime: Runtime[UserContext],
) -> dict:
    update = dict(state.get("preference_update") or {})
    if update:
        runtime.store.put(USER_MEMORY_NS, runtime.context["user_id"], update)
    return {}


def load_user_preference(
    state: LongTermState,
    runtime: Runtime[UserContext],
) -> dict:
    item = runtime.store.get(USER_MEMORY_NS, runtime.context["user_id"])
    return {"loaded_preference": dict(item.value) if item else {}}


def render_preference(state: LongTermState) -> dict:
    preference = state.get("loaded_preference") or {}
    if not preference:
        return {"response": "未找到该用户的长期偏好；使用系统默认输出。"}
    return {
        "response": (
            f"读取长期偏好：language={preference.get('language')}; "
            f"answer_style={preference.get('answer_style')}"
        )
    }


long_builder = StateGraph(LongTermState, context_schema=UserContext)
long_builder.add_node("persist_user_preference", persist_user_preference)
long_builder.add_node("load_user_preference", load_user_preference)
long_builder.add_node("render_preference", render_preference)
long_builder.add_edge(START, "persist_user_preference")
long_builder.add_edge("persist_user_preference", "load_user_preference")
long_builder.add_edge("load_user_preference", "render_preference")
long_builder.add_edge("render_preference", END)

long_store = InMemoryStore()
long_checkpointer = InMemorySaver()
long_app = long_builder.compile(
    checkpointer=long_checkpointer,
    store=long_store,
)

print("long-term store =", type(long_store).__name__)


## 4. 长期记忆对照：同 user 跨 thread 保留，不同 user 隔离


In [ ]:
USER_A = {"user_id": "customer-A"}
USER_B = {"user_id": "customer-B"}

# thread A-1 写入用户 A 的长期偏好
written = long_app.invoke(
    {
        "preference_update": {
            "language": "中文",
            "answer_style": "先列证据缺口",
        }
    },
    {"configurable": {"thread_id": "preference-A-1"}},
    context=USER_A,
)
print("写入后:", written["response"])

# 新 thread A-2，同一个 user_id：仍能读取 Store
same_user_new_thread = long_app.invoke(
    {},
    {"configurable": {"thread_id": "preference-A-2"}},
    context=USER_A,
)
print("同 user / 新 thread:", same_user_new_thread["response"])

# 新 thread B-1，不同 user_id：不得读取用户 A 的偏好
different_user = long_app.invoke(
    {},
    {"configurable": {"thread_id": "preference-B-1"}},
    context=USER_B,
)
print("不同 user:", different_user["response"])

stored_item = long_store.get(USER_MEMORY_NS, USER_A["user_id"])
print("Store item:", stored_item.value)

assert same_user_new_thread["loaded_preference"]["language"] == "中文"
assert same_user_new_thread["loaded_preference"]["answer_style"] == "先列证据缺口"
assert different_user["loaded_preference"] == {}
assert stored_item.value["language"] == "中文"
print("04 long-term memory ok")


## 5. 通关检查与生产边界

| 检查 | 合格证据 |
|---|---|
| 短期记忆 | 同 `thread_id` 的 checkpoint 含多轮 messages；新 thread 不带旧消息 |
| 长期记忆 | 同 `user_id` 换 thread 仍能读取 Store；不同 user 为空 |
| 权威性 | 长期记忆只保存偏好或经批准的摘要，不保存为“当前有效规则” |
| 跨重启耐久 | 课堂 In-memory 后端不满足；生产应选择数据库支持的 checkpointer 与 Store，并定义权限、保留期、更新和删除策略 |

API 与具体后端以课程锁定依赖和当前 LangGraph 官方文档为准。


In [ ]:
assert type(short_checkpointer).__name__ == "InMemorySaver"
assert type(long_store).__name__ == "InMemoryStore"
assert "checkpoint_id" in snapshot.config["configurable"]
assert long_store.get(USER_MEMORY_NS, "customer-B") is None

print("04 memory demo ok")
